# Text Analizer

## Import libraries

In [108]:
import time
import spacy
import sqlite3
import lemminflect

In [109]:
NLP = spacy.load("en_core_web_sm", exclude = ['parser', 'ner'])

## Connect to database

In [110]:
DATABASE_FILENAME = 'word_cefr_minified.db'

conn = sqlite3.connect(DATABASE_FILENAME)
cursor = conn.cursor()

## Calculate functions and variables

In [111]:
ABBREVIATION_MAPPING = {
    "'m": "am",
    "'s": "is",
    "'re": "are",
    "'ve": "have",
    "'d": "had",
    "n't": "not",
    "'ll": "will"
}

DIFFICULTY_MAPPING_REVERSE = {
    1: 'A1',
    2: 'A2',
    3: 'B1',
    4: 'B2',
    5: 'C1',
    6: 'C2'
}


def is_punctuation(word: str) -> bool:
    return not word and not any(char.isalpha() for char in word)


def custom_tokenize_text(text: str) -> list[tuple[str, str, str]]:
    text = text.replace("’", "'")
    tokens = []
    doc = NLP(text)
    for token in doc:
        word = token.text.lower().strip()
        word_pos = token.tag_
        proposed_lemma = token._.lemma().lower()

        abbreviation_form = ABBREVIATION_MAPPING.get(word)
        if abbreviation_form:
            word = abbreviation_form
            lemma = word
        elif proposed_lemma is None:
            lemma = word.lower()
        else:
            lemma = proposed_lemma

        tokens.append((word, lemma, word_pos))

    return tokens


def fetch_word_pos_level_tokens(word_pos_tokens_set: set[tuple[str, str]]) -> dict[tuple[str, str], float]:
    placeholders = ','.join(['(?, ?)' for _ in range(len(word_pos_tokens_set))])

    cursor.execute('''
        WITH word_pos_tags(word, pos_tag) AS (
            VALUES {}
        )
        SELECT
            word_pos_tags.word,
            word_pos_tags.pos_tag,
            COALESCE(
                AVG(CASE WHEN pt.tag = word_pos_tags.pos_tag THEN wp.level END),
                AVG(wp.level)
            ) AS avg_level
        FROM word_pos_tags
        JOIN words w ON word_pos_tags.word = w.word
        JOIN word_pos wp ON w.word_id = wp.word_id
        JOIN pos_tags pt ON wp.pos_tag_id = pt.tag_id
        GROUP BY word_pos_tags.word, word_pos_tags.pos_tag
    '''.format(placeholders), [item for sublist in word_pos_tokens_set for item in sublist])

    word_pos_level_tokens = cursor.fetchall()

    return {(word, pos_tag): float(avg_level) for word, pos_tag, avg_level in word_pos_level_tokens}


def get_word_pos_tokens_set(tokens: list[tuple[str, str, str]]) -> set[tuple[str, str]]:
    return {(token[0], token[2]) for token in tokens if not is_punctuation(token[1])}


def get_levels_tokens(tokens: list[tuple[str, str, str]]) -> list[tuple[str, str, str, float]]:
    word_pos_set = get_word_pos_tokens_set(tokens)
    word_pos_unique_level_tokens = fetch_word_pos_level_tokens(word_pos_set)

    word_pos_level_tokens = []
    for token in tokens:
        word, lemma, word_pos = token

        level = word_pos_unique_level_tokens.get((word, word_pos))
        if level is None:
            level = 0

        word_pos_level_tokens.append((word, lemma, word_pos, level))

    return word_pos_level_tokens


def get_word_level_count_statistic(level_tokens: list[tuple[str, str, str, float]]) -> list[int]:
    difficulty_levels_count = [0] * 6
    for token in level_tokens:
        level = round(token[3])
        if level:
            difficulty_levels_count[level - 1] += 1

    return difficulty_levels_count


def get_word_level_count_statistic_unique(level_tokens: list[tuple[str, str, str, float]]) -> list[int]:
    processed_word_pos_set = set()
    difficulty_levels_count = [0] * 6
    for token in level_tokens:
        level = round(token[3])
        to_check_tuple = (token[0], token[2])
        if level and not to_check_tuple in processed_word_pos_set:
            processed_word_pos_set.add(to_check_tuple)
            difficulty_levels_count[level - 1] += 1

    return difficulty_levels_count


def get_not_found_words(level_tokens: list[tuple[str, str, str, float]]) -> set[str]:
    not_found_words = set()
    for token in level_tokens:
        if not token[3] and token[0] and all(char.isalpha() for char in token[0]):
            not_found_words.add(token[0])

    return not_found_words


def filter_for_desired_level(level_tokens: list[tuple[str, str, str, float]],
                            min_level: float, max_level: float = 6) -> set[tuple[str, str, str, float]]:
    filtered_tokens = set()
    for token in level_tokens:
        level = token[3]
        if level >= min_level and level <= max_level:
            filtered_tokens.add(token)

    return filtered_tokens


In [112]:
# Source: ChatGPT 3.5
input_text = """
Philosophy (which means "love of wisdom" in old Greek) is the careful study of big and basic questions about life, reason, knowledge, value, mind, and language. It is a way of thinking that asks about its own steps and ideas.
In the past, many sciences, such as physics and the study of the mind, were part of philosophy. But today, they are seen as separate school subjects. Important traditions in the history of thought include Western, Arabic–Persian, Indian, and Chinese philosophy. Western philosophy started in Ancient Greece and covers many areas. A main topic in Arabic–Persian thought is the link between reason and revelation. Indian philosophy joins the spiritual question of how to reach light with the study of reality and ways of learning. Chinese thought looks mostly at daily life issues about good social rules, government, and how to grow oneself.
The main parts of philosophy are the study of knowledge, ethics, logic, and metaphysics. The study of knowledge asks what knowledge is and how to get it. Ethics looks at moral rules and what makes good action. Logic is the study of correct reasoning and how good arguments are different from bad ones. Metaphysics looks at the most general parts of reality, life, objects, and their features. Other areas include art, language, mind, religion, science, numbers, history, and politics. In each part, there are different schools of thought that support different rules, ideas, or ways.
Philosophers use many ways to reach knowledge. They include clear thinking, use of common sense, simple tests in the mind, looking at daily language, description of experience, and asking hard questions. Philosophy is also linked to many other areas, such as science, numbers, work, law, and news writing. It gives a view that brings many fields together and studies the key ideas of these areas. It also looks at their steps and moral results.
"""

In [113]:
start_time_nlp = time.time()
tokens = custom_tokenize_text(input_text)

print("NLP:", round((time.time() - start_time_nlp) * 1000), "ms")

start_time_cefr = time.time()
level_tokens = get_levels_tokens(tokens)
print("CEFR levels:", round((time.time() - start_time_cefr) * 1000), "ms")

print('-' * 30)

print("Text length:", len(input_text))
print("Total tokens:", len(tokens))

NLP: 35 ms
CEFR levels: 37 ms
------------------------------
Text length: 1897
Total tokens: 385


In [114]:
counter = 0
print(f'{"WORD".ljust(26)}\t{"LEMMA".ljust(26)}\tPOS\tLEVEL\tCEFR')
print('-' * 85)
for token in level_tokens:
    word, lemma, pos, level = token
    if pos != '_SP':
        print(f'{word.ljust(26)}\t{lemma.ljust(26)}\t{pos}\t{"{:.2f}".format(level)}\t{DIFFICULTY_MAPPING_REVERSE.get(round(level))}')

        counter += 1
        if counter >= 200:
            break

WORD                      	LEMMA                     	POS	LEVEL	CEFR
-------------------------------------------------------------------------------------
philosophy                	philosophy                	NNP	3.00	B1
(                         	(                         	-LRB-	0.00	None
which                     	which                     	WDT	1.00	A1
means                     	mean                      	VBZ	4.00	B2
"                         	"                         	``	0.00	None
love                      	love                      	NN	1.00	A1
of                        	of                        	IN	1.00	A1
wisdom                    	wisdom                    	NN	2.00	A2
"                         	"                         	''	0.00	None
in                        	in                        	IN	1.00	A1
old                       	old                       	JJ	1.00	A1
greek                     	greek                     	NNP	2.96	B1
)                         	)                        

In [115]:
difficulty_levels_count = get_word_level_count_statistic(level_tokens)

print('CEFR statistic (total words):')
for i in range(1, 7):
    print(f'{DIFFICULTY_MAPPING_REVERSE.get(i)}: {difficulty_levels_count[i - 1]}')

print("Proportion of under A2 level words:", (difficulty_levels_count[0] + difficulty_levels_count[1]) / sum(difficulty_levels_count) if sum(difficulty_levels_count) > 0 else 0)

CEFR statistic (total words):
A1: 195
A2: 74
B1: 32
B2: 8
C1: 2
C2: 2
Proportion of under A2 level words: 0.8594249201277955


In [116]:
difficulty_levels_count_unique = get_word_level_count_statistic_unique(level_tokens)

print('CEFR statistic (unique words):')
for i in range(1, 7):
    print(f'{DIFFICULTY_MAPPING_REVERSE.get(i)}: {difficulty_levels_count_unique[i - 1]}')

CEFR statistic (unique words):
A1: 82
A2: 45
B1: 23
B2: 7
C1: 2
C2: 2


In [117]:
not_found_words_set = get_not_found_words(level_tokens)

not_found_words_list = list(not_found_words_set)
not_found_words_list.sort()

print('Not found words:', len(not_found_words_list))

if len(not_found_words_list):
    print('\n'.join(not_found_words_list))

Not found words: 0


In [118]:
desired_level_words_set = filter_for_desired_level(level_tokens, 4)

desired_level_words_list = list(desired_level_words_set)
desired_level_words_list.sort(key=lambda x: (x[2], x[0]))

print('\tWords with level B2 and higher:', len(desired_level_words_list))
for word_data in desired_level_words_list:
    word, _, pos, level = word_data
    print(word.lower().ljust(26), pos.ljust(6), "{:.2f}".format(level).ljust(6), DIFFICULTY_MAPPING_REVERSE.get(round(level)))

	Words with level B2 and higher: 10
moral                      JJ     4.00   B2
persian                    JJ     5.41   C1
western                    JJ     4.00   B2
revelation                 NN     4.00   B2
metaphysics                NNP    6.00   C2
persian                    NNP    5.04   C1
western                    NNP    4.00   B2
metaphysics                NNS    6.00   C2
learning                   VBG    4.00   B2
means                      VBZ    4.00   B2


In [119]:
conn.close()